# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring, which feeds a Ranking.**

The decision I care about is: which pages should a reviewer look at first? This kind of question
is a scoring/ranking problem, not a classification problem

The model gives one number, a score, to every page. Then I sort pages by that score to build the
review list. Behind the scenes, that score can come from a classifier that guesses if a page is
likely declining (more on this in Section 2). But the reviewer does not need a single yes/no
answer for each page. They need the order of the list. That is why I call this scoring/ranking
and not plain classification

It is not clustering, because I already know what I want to predict: decline. It is not plain
classification either, because what matters at the end is a short, ordered list, not 30,000
separate yes/no calls

In [12]:
# Section 1 :  writing the framing choice so it is clear
lane = "Lane 2: Refresh / Content Opportunity Scoring"
task_type = "Scoring (feeds a Ranking)"
decision = "Which pages should a content reviewer look at first?"

print(f"Lane: {lane}")
print(f"Task type: {task_type}")
print(f"Decision this supports: {decision}")

Lane: Lane 2: Refresh / Content Opportunity Scoring
Task type: Scoring (feeds a Ranking)
Decision this supports: Which pages should a content reviewer look at first?


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label` : a proxy, not an observed future outcome.**

`is_declining_label = 1` when `trend_direction == "down"`, meaning impressions in the last 30 days
dropped more than 20% versus the previous 30 days. This column already exists in the raw CSV as
`trend_direction` / `trend_pct`

This is a proxy, not a result I can call "observed" in the strict sense. It only compares two
recent, already elapsed 30-day windows inside the same 90-day export — it does not tell me if a
page will actually recover after someone refreshes it (that needs a later, held-out window I don't
have in the starter data). So my honest claim is: I can rank pages by how closely they match a
recent-decline pattern, not predict what a refresh will achieve.

Per the data dictionary, I must never use `trend_direction` or `trend_pct` as a model **feature** —
only as the label. Using it as a feature would let the model look up the answer instead of learning
it from the other signals (search visibility, freshness, position, content depth, engagement).

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label` — a proxy, not a obsereved outcome ( real future outcome )**

`is_declining_label = 1` when `trend_direction == "down"` : This means impressions dropped more
than 20% in the last 30 days compared to the 30 days before that. This column already exists in
the raw CSV, as `trend_direction` and `trend_pct`

This is a proxy, not a true observed outcome. It only compares two recent 30-day windows inside
the same 90-day file. It does not tell me if a page will actually get better after someone
refreshes it :  that would need a later window(held out window), one I don't have yet. So my claim stays careful: I
can rank pages by how much they look like a recent decline. I cannot say a refresh will fix them

One rule to remember: `trend_direction` and `trend_pct` can be the LABEL (model feature). They can never be a
FEATURE. If I use them as a feature, the model just copies the answer instead of learning it from
other signals like search visibility ,traffic, freshness, position, and content length ,engagement

In [13]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# The proxy target already lives in the raw column trend_direction
# NEVER use trend_direction / trend_pct as a FEATURE later : only as this label
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows: {len(df):,}")
print(f"Positive rate (Declining With Demand proxy): {df['is_declining_label'].mean() * 100:.2f}%")

df[["content_id", "trend_pct", "trend_direction", "is_declining_label"]].head(10)

Rows: 30,000
Positive rate (Declining With Demand proxy): 54.21%


,content_id,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,-41.4,down,1
1,content_a1fb4e703a9e,-57.7,down,1
2,content_9aa793d4d895,-60.9,down,1
3,content_331d6c4de07b,-13.8,stable,0
4,content_d99b7a2d90ca,-34.7,down,1
5,content_d4084a4bc775,-38.9,down,1
6,content_9a34b442b552,-92.3,down,1
7,content_a63219c6e95a,0.6,stable,0
8,content_5e6c160719bc,-58.8,down,1
9,content_c27558df2b0c,-29.2,down,1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@50** (We will also check : Precision@20 and @100, plus average precision=>  as supporting checks).

A reviewer only has time for a short list, so I judge quality at the TOP of the queue, not accuracy
across all 30,000 pages. Precision@50 asks: of the 50 pages my score ranks highest, what share are
actually flagged as declining-with-demand (my proxy label) (how many are actually flagged as declining) ?

"Good" means clearly above the base decline rate of the data :  54.21% of all pages carry the proxy
label (computed below or flagged as declining) . If my top 50 (Precision@50) lands at or below that number, my ranking is no better than
grabbing 50 pages at random and it has not earned the "ML" label. I also want this measured on
held-out clients (which the model has not seen before), not rows the model already trained on, because pages from the same client behave
similarly and a client-blind split checks whether the pattern actually generalizes

In [14]:
import numpy as np


def precision_at_k(labels: pd.Series, scores: pd.Series, k: int) -> float:
    """Share of true positives among the top k ranked items."""
    order = np.argsort(-scores.to_numpy())
    top_k_labels = labels.to_numpy()[order][:k]
    return float(top_k_labels.mean())


base_rate = df["is_declining_label"].mean()
print(f"Base decline rate (what 50 random pages would score): {base_rate:.3f}")
print("Any real ranking method must clear this number at Precision@50 to be worth using")

Base decline rate (what 50 random pages would score): 0.542
Any real ranking method must clear this number at Precision@50 to be worth using


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one pseudonymized content item (one web page), for one client, measured over a
trailing 90day window**

The starter file `content_refresh_anonymized.csv` has 30,000 such rows across 32 clients.
`content_id` is the unique key per page. `client_id` groups pages by account and is used only for
grouped train/test splits later, never as a feature

In [15]:
# The lane's slice of the starter data :  one row = one pseudonymized page
lane_columns = [
    "content_id", "client_id", "content_type", "impressions_90d",
    "avg_position", "ctr", "days_since_last_update", "word_count",
    "trend_direction", "trend_pct", "is_declining_label",
]

lane_df = df[lane_columns].copy()

print(f"Shape: {lane_df.shape[0]:,} rows x {lane_df.shape[1]} columns")
print(f"Unique content_id: {lane_df['content_id'].nunique():,}  (matches row count -> one row per page)")
print(f"Unique client_id: {lane_df['client_id'].nunique()}")

lane_df.head(5)

Shape: 30,000 rows x 11 columns
Unique content_id: 30,000  (matches row count -> one row per page)
Unique client_id: 32


,content_id,client_id,content_type,impressions_90d,avg_position,ctr,days_since_last_update,word_count,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,0.76,20,3221.0,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,0.05,25,2481.0,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,0.09,20,3515.0,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,0.49,22,NaN,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,0.13,14,2803.0,down,-34.7,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A fixed rule can't hold all these signals at once : the code below proves it**

I sorted the pages by one signal at a time :  freshness, visibility, position, CTR  : and checked
Precision@50 for each. None of them beat the base decline rate of 54.2% :  one of them (best position
first) barely clears 16%. A page can be old AND have no traffic (not worth reviewing) or fresh
AND declining (a freshness rule misses it) or well positioned AND about to drop (a position rule
misses it too). The signals only mean something in combination and the right combination is not
obvious by eye

Even a hand built rule that DOES combine several signals :  this project's own baseline in
`scripts/02_baseline_score.py`, which blends visibility + freshness + position + depth :  reports
Precision@50 = 0.240 in this repo's own `GUIDE.md`. That is still below the 54.2% base rate, because
it was tuned for general "refresh worthiness" not for this specific decline pattern. Getting the
weights and interactions right, from data instead of guesswork, is exactly the job a scoring model
does :  and exactly what a single if statement cannot

In [16]:
single_signal_checks = [
    ("days_since_last_update", False, "stalest pages first"),
    ("impressions_90d", False, "most visible pages first"),
    ("avg_position", True, "best search position first (excluding no data rows)"),
    ("ctr", True, "worst click-through-rate first"),
]

print(f"{'Signal':45}{'Precision@50':>14}")
for column, ascending, label_text in single_signal_checks:
    # avg_position of 0 means "no data", not rank zero :  drop it before sorting
    subset = df[df["avg_position"] > 0] if column == "avg_position" else df
    ranked = subset.sort_values(column, ascending=ascending)
    score = ranked["is_declining_label"].head(50).mean()
    print(f"{label_text:45}{score:>14.3f}")

print(f"{'BASE RATE (random 50 pages)':45}{base_rate:>14.3f}")
print()
print("Every single signal rule sits AT OR BELOW the base rate ")
print("None of them, alone, finds decliners better than chance ")

Signal                                         Precision@50
stalest pages first                                   0.520
most visible pages first                              0.420
best search position first (excluding no data rows)         0.180
worst click-through-rate first                        0.620
BASE RATE (random 50 pages)                           0.542

Every single signal rule sits AT OR BELOW the base rate 
None of them, alone, finds decliners better than chance 


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.